# Polymarket API — Auth & balance test

Uses the same authentication as `live_execution.py`: load credentials from `.env`, create `ClobClient`, then derive or set API creds. Finally fetches USDC collateral balance.

**`.env` format:** Use `POLYMARKET_PRIVATE_KEY=0xYour64HexDigits` with **no quotes** and no spaces around `=`. If you get "Non-hexadecimal digit found", the key string has an invalid character (often from quotes or a stray newline).

In [ ]:
# Ensure project root is on path when running from analytics/
import sys
from pathlib import Path
_root = Path.cwd().parent if Path.cwd().name == "analytics" else Path.cwd()
sys.path.insert(0, str(_root))

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from py_clob_client.client import ClobClient
from py_clob_client.clob_types import ApiCreds, AssetType, BalanceAllowanceParams

# Load .env from project root or config/; override=True so we always use latest .env
_root = Path.cwd().parent if Path.cwd().name == "analytics" else Path.cwd()
for _env_path in [_root / "config" / ".env", _root / ".env", Path.cwd() / ".env"]:
    if _env_path.exists():
        load_dotenv(dotenv_path=_env_path, override=True)
        break
else:
    load_dotenv(override=True)
host = os.getenv("POLYMARKET_HOST", "https://clob.polymarket.com")
chain_id = int(os.getenv("POLYMARKET_CHAIN_ID", "137"))
private_key = os.getenv("POLYMARKET_PRIVATE_KEY", "").strip()
# Strip surrounding quotes (e.g. .env with POLYMARKET_PRIVATE_KEY="0x..." or '0x...')
if len(private_key) >= 2 and private_key[0] == private_key[-1] and private_key[0] in '"\'':
    private_key = private_key[1:-1].strip()
# Remove any internal whitespace/newlines (invalid in hex keys)
private_key = "".join(private_key.split())
funder = os.getenv("POLYMARKET_FUNDER_ADDRESS", "").strip()
if len(funder) >= 2 and funder[0] == funder[-1] and funder[0] in '"\'':
    funder = funder[1:-1].strip()
signature_type = int(os.getenv("POLYMARKET_SIGNATURE_TYPE", "1"))

if not private_key:
    raise RuntimeError("Missing POLYMARKET_PRIVATE_KEY in .env")
if not funder:
    raise RuntimeError("Missing POLYMARKET_FUNDER_ADDRESS in .env")
# Private key must be 0x + 64 hex chars (or 64 hex chars only)
hex_part = private_key[2:] if private_key.lower().startswith("0x") else private_key
if len(hex_part) != 64 or not all(c in "0123456789abcdefABCDEF" for c in hex_part):
    raise RuntimeError(
        "POLYMARKET_PRIVATE_KEY must be 64 hex digits, optionally prefixed with 0x. "
        "Check .env for extra quotes, spaces, or newlines (e.g. use POLYMARKET_PRIVATE_KEY=0x... with no quotes)."
    )

auth_client = ClobClient(
    host=host,
    chain_id=chain_id,
    key=private_key,
    signature_type=signature_type,
    funder=funder,
)

api_key = os.getenv("POLYMARKET_API_KEY", "").strip()
api_secret = os.getenv("POLYMARKET_API_SECRET", "").strip()
api_passphrase = os.getenv("POLYMARKET_API_PASSPHRASE", "").strip()
if api_key and api_secret and api_passphrase:
    auth_client.set_api_creds(ApiCreds(api_key=api_key, api_secret=api_secret, api_passphrase=api_passphrase))
else:
    creds = auth_client.create_or_derive_api_creds()
    auth_client.set_api_creds(creds)
    print("[auth] Derived API creds from private key.")

print("Authenticated.")

[auth] Derived API creds from private key.
Authenticated.


In [2]:
# Pass signature_type so the API returns balance for your wallet type (1 = Email/Magic).
# Without this, the API can return 0 for proxy/ Magic wallets.
balance = auth_client.get_balance_allowance(
    BalanceAllowanceParams(asset_type=AssetType.COLLATERAL, signature_type=signature_type)
)
raw = balance.get("balance") if isinstance(balance, dict) else getattr(balance, "balance", None)
usdc_balance = int(raw) / 1e6
print(f"USDC Balance: ${usdc_balance:.2f}")

USDC Balance: $339.54
